# Kronos Forex Forecasting on Google Colab

**Purpose:** Run the pre-trained [Kronos](https://github.com/shiyu-coder/Kronos) financial foundation model on your own Forex OHLCV CSV data.

### Pipeline
`Forex CSV → clean/normalize columns → load Kronos-small → forecast future candles → chart → save predictions`

> This notebook is for **forecasting/research**, not an automatic trading system. A price forecast is not a guaranteed buy/sell signal.

## 1. Enable GPU

In Colab go to:

**Runtime → Change runtime type → Hardware accelerator → GPU**

Then run the next cell.

In [ ]:
import os, sys, subprocess, textwrap, json, math
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Kronos will run much slower on CPU.")

In [ ]:
# Clone the official Kronos repository
%cd /content
!rm -rf Kronos
!git clone -q https://github.com/shiyu-coder/Kronos.git
%cd /content/Kronos

# Install the repository dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify the Kronos installation
import sys
sys.path.insert(0, "/content/Kronos")

from model import Kronos, KronosTokenizer, KronosPredictor
print("Kronos imports: OK")

## 2. Upload your Forex CSV

Your CSV should contain at least:

- `open`
- `high`
- `low`
- `close`

Recommended:

- `volume`
- `timestamp` / `timestamps` / `time` / `datetime`

The notebook tries to automatically recognize common column names and convert them to Kronos' expected format.

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No CSV uploaded.")

csv_name = next(iter(uploaded.keys()))
df_raw = pd.read_csv(csv_name)

print("Loaded:", csv_name)
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
display(df_raw.head())

In [ ]:
# Clean and standardize Forex data

def standardize_forex_dataframe(df):
    df = df.copy()
    original_cols = list(df.columns)

    # Case-insensitive aliases
    aliases = {
        "timestamp": ["timestamp", "timestamps", "time", "datetime", "date", "gmt_time"],
        "open": ["open", "o"],
        "high": ["high", "h"],
        "low": ["low", "l"],
        "close": ["close", "c"],
        "volume": ["volume", "vol", "tick_volume", "tickvolume"],
        "amount": ["amount", "turnover", "quote_volume"]
    }

    lower_map = {str(c).strip().lower(): c for c in df.columns}

    rename = {}
    for target, candidates in aliases.items():
        for candidate in candidates:
            if candidate in lower_map:
                rename[lower_map[candidate]] = target
                break

    df = df.rename(columns=rename)

    required = ["open", "high", "low", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Your CSV must contain OHLC columns."
        )

    # Timestamp is strongly recommended for correct future time features.
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.dropna(subset=["timestamp"])
        df = df.sort_values("timestamp")
        df = df.drop_duplicates(subset=["timestamp"], keep="last")
    else:
        print("WARNING: No timestamp column found. A synthetic time index will be created.")

    # Numeric OHLCV columns
    for c in ["open", "high", "low", "close", "volume", "amount"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=required)

    # Kronos accepts missing volume/amount. We make them explicit.
    if "volume" not in df.columns:
        df["volume"] = 0.0

    if "amount" not in df.columns:
        # For Forex, true traded amount is usually unavailable from broker OHLC data.
        # Use volume * typical price as a proxy only.
        typical_price = (df["open"] + df["high"] + df["low"] + df["close"]) / 4.0
        df["amount"] = df["volume"] * typical_price

    # Replace invalid values
    numeric_cols = ["open", "high", "low", "close", "volume", "amount"]
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["open", "high", "low", "close"])

    if len(df) < 50:
        raise ValueError(f"Only {len(df)} usable rows remain. Kronos needs a meaningful history.")

    return df.reset_index(drop=True)

df = standardize_forex_dataframe(df_raw)

print("Cleaned shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())
display(df.tail())

## 3. Configure the forecast (Full Month September 2026 Backtest)

### Why August 31 Cutoff?
- **`CUTOFF_DATE = "2026-08-31 23:59:59"`**: Kronos ko sirf 31 August tak ka data milega.
- **`LOOKBACK = 512`**: August aur July ki 512 candles ka historical context.
- **`PRED_LEN = 504`**: September 2026 mein **exact 504 H1 candles** hain. Kronos poora September predict karega.
- **Full Ground-Truth Verification**: Hamare CSV aur MT5 mein September ki saari 504 candles pehle se mojood hain, to hum 1-to-1 match kar sakte hain!

In [ ]:
# ===== USER SETTINGS (Full September 2026 Test) =====

# 1. Cutoff: 31 August 2026 (Kronos will NOT see anything from September onwards)
CUTOFF_DATE = "2026-08-31 23:59:59"

# 2. Context Window & Forecast Horizon
LOOKBACK = 512       # Context: August 2026 ki 512 candles
PRED_LEN = 504       # Exactly 504 H1 candles in September 2026 (Full Month)

MODEL_NAME = "NeoQuasar/Kronos-small"
TOKENIZER_NAME = "NeoQuasar/Kronos-Tokenizer-base"

# For 504 steps, temperature 0.7 gives stable, realistic paths without extreme noise
TEMPERATURE = 0.7
TOP_P = 0.9
SAMPLE_COUNT = 1     # 1 is fast; 3 gives averaged ensemble path

# ====================================================
print(f"Cutoff Date: {CUTOFF_DATE}")
print(f"Context Lookback: {LOOKBACK} candles (July-August 2026)")
print(f"Forecast Horizon: {PRED_LEN} candles (Full September 2026)")
print(f"Model: {MODEL_NAME}")

## 4. Load the pre-trained Kronos model

The first run downloads the tokenizer/model from Hugging Face. After that, Colab can reuse the local cache during the session.

In [ ]:
import torch
from model import Kronos, KronosTokenizer, KronosPredictor

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = KronosTokenizer.from_pretrained(TOKENIZER_NAME)
model = Kronos.from_pretrained(MODEL_NAME)

predictor = KronosPredictor(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_context=512
)

print("Kronos model loaded successfully.")

In [ ]:
# Build historical input, actual future (if in CSV), and forecast timestamps
feature_cols = ["open", "high", "low", "close", "volume", "amount"]

# 1. Apply CUTOFF_DATE if specified
if CUTOFF_DATE is not None and "timestamp" in df.columns:
    cutoff_ts = pd.to_datetime(CUTOFF_DATE)
    history_df = df[df["timestamp"] <= cutoff_ts].copy()
    actual_future_df = df[df["timestamp"] > cutoff_ts].copy()
    print(f"Applied CUTOFF_DATE: {CUTOFF_DATE}")
    print(f"Historical candles available before cutoff: {len(history_df)}")
    if len(actual_future_df) > 0:
        print(f"Actual market candles in October available in CSV: {len(actual_future_df)} (from {actual_future_df['timestamp'].iloc[0]} to {actual_future_df['timestamp'].iloc[-1]})")
else:
    history_df = df.copy()
    actual_future_df = pd.DataFrame()

if len(history_df) < LOOKBACK:
    raise ValueError(
        f"Data before cutoff has only {len(history_df)} rows, but LOOKBACK={LOOKBACK}. "
        f"Reduce LOOKBACK or check your CUTOFF_DATE."
    )

x_df = history_df.iloc[-LOOKBACK:][feature_cols].copy()

if "timestamp" in history_df.columns:
    x_timestamp = history_df.iloc[-LOOKBACK:]["timestamp"].copy()
    deltas = x_timestamp.diff().dropna()
    if len(deltas) == 0:
        raise ValueError("Could not infer timestamp frequency.")
    interval = deltas.median()
    if pd.isna(interval) or interval <= pd.Timedelta(0):
        raise ValueError("Invalid timestamp interval.")

    last_ts = x_timestamp.iloc[-1]
    y_timestamp = pd.Series(
        [last_ts + interval * i for i in range(1, PRED_LEN + 1)]
    )
else:
    x_timestamp = pd.date_range(start=pd.Timestamp("2000-01-01"), periods=LOOKBACK, freq="min")
    y_timestamp = pd.date_range(start=x_timestamp[-1] + pd.Timedelta(minutes=1), periods=PRED_LEN, freq="min")

print(f"Historical context: {len(x_df)} candles (from {x_timestamp.iloc[0]} to {x_timestamp.iloc[-1]})")
print(f"Forecast horizon: {len(y_timestamp)} candles (~1 Month)")
print(f"Forecast start: {y_timestamp.iloc[0]}")
print(f"Forecast end:   {y_timestamp.iloc[-1]}")

## 5. Run Kronos forecast

This is the main inference step.

Kronos predicts future **OHLCV-style candles**, not directly `BUY` or `SELL`.

We will derive a simple directional interpretation from predicted close prices afterward.

In [ ]:
import time

start = time.time()

pred_df = predictor.predict(
    df=x_df,
    x_timestamp=x_timestamp,
    y_timestamp=y_timestamp,
    pred_len=PRED_LEN,
    T=TEMPERATURE,
    top_p=TOP_P,
    sample_count=SAMPLE_COUNT,
    verbose=True
)

elapsed = time.time() - start

print(f"Forecast completed in {elapsed:.2f} seconds")
display(pred_df.head(10))

In [ ]:
# Display forecast summary

last_close = float(x_df["close"].iloc[-1])
first_pred_close = float(pred_df["close"].iloc[0])
last_pred_close = float(pred_df["close"].iloc[-1])

first_change_pct = (first_pred_close / last_close - 1.0) * 100
last_change_pct = (last_pred_close / last_close - 1.0) * 100

print(f"Last historical close: {last_close:.5f}")
print(f"First predicted close:  {first_pred_close:.5f}")
print(f"Final predicted close:  {last_pred_close:.5f}")
print(f"First predicted change: {first_change_pct:+.3f}%")
print(f"Final predicted change: {last_change_pct:+.3f}%")

if last_pred_close > last_close:
    print("Directional forecast: UP / bullish bias")
elif last_pred_close < last_close:
    print("Directional forecast: DOWN / bearish bias")
else:
    print("Directional forecast: FLAT")

## 6. Visualize historical close vs Kronos forecast

In [ ]:
import matplotlib.pyplot as plt

history = x_df.copy()
history.index = x_timestamp

plt.figure(figsize=(16, 7))
# 1. Historical Close (August Context)
plt.plot(history.index, history["close"], label="Historical Context (August 2026)", color="#1f77b4", linewidth=1.5)

# 2. Actual Future Candles (Real September market in MT5)
if not actual_future_df.empty:
    actual_sub = actual_future_df.iloc[:PRED_LEN]
    plt.plot(actual_sub["timestamp"], actual_sub["close"], label=f"Actual Market (Real September 2026: {len(actual_sub)} candles)", color="#2ca02c", linestyle="--", linewidth=2.0)

# 3. Kronos Forecast for September
forecast_index = y_timestamp.values if "timestamp" in df.columns else pred_df.index
plt.plot(forecast_index, pred_df["close"], label=f"Kronos AI Forecast (Full September 2026)", color="#ff7f0e", linewidth=2.0)

plt.axvline(history.index[-1], color="black", linestyle=":", label="Cutoff Date (31 August Close)")
plt.title("Kronos AI 1-Month September Forecast vs Real September MT5 Market", fontsize=14, fontweight="bold")
plt.xlabel("Date and Time", fontsize=12)
plt.ylabel("Price (USD)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Forecasted OHLC table

The model gives a complete future candle structure:

`open, high, low, close, volume, amount`

For trading research, the most important first step is usually comparing predicted close/returns with the actual future close.

In [ ]:
display(pred_df.round(6))

In [ ]:
# Optional: simple forecast-return columns for research

result = pred_df.copy()
result["pred_return_from_previous"] = result["close"].pct_change()
result["pred_return_from_last_actual"] = result["close"] / last_close - 1.0
result["pred_direction"] = np.where(
    result["close"] > last_close, "UP",
    np.where(result["close"] < last_close, "DOWN", "FLAT")
)

display(result.round(6))

## 8. Save predictions

This creates a CSV that you can download and later use for backtesting, dashboards, or a trading research pipeline.

In [ ]:
output_path = "/content/kronos_forex_predictions.csv"
result.to_csv(output_path, index_label="timestamp")

print("Saved:", output_path)

from google.colab import files
files.download(output_path)

# Important: Forex usage notes

### 1. Kronos is not a trading strategy
Kronos forecasts future financial K-lines. It does **not** inherently know your risk rules, spread, swap, slippage, stop-loss, take-profit, leverage, or broker execution.

### 2. Forex data quality matters
For XAUUSD/Forex, use consistent broker data and keep the timezone/timeframe consistent.

### 3. Volume is different in Forex
Many MT5 Forex datasets contain **tick volume**, not centralized exchange-traded volume. The notebook therefore treats missing volume/amount conservatively.

### 4. Do not judge the model from one forecast
For serious evaluation, use walk-forward testing:

`train/history → forecast → reveal actual candles → calculate error → move forward → repeat`

Useful metrics include MAE, RMSE, directional accuracy, return correlation, and trading metrics after adding realistic spread/slippage.

### 5. Next step for your trading project
After this notebook works, the stronger architecture is:

`MT5 XAUUSD data → Kronos forecast → feature/signal layer → risk manager → backtest → paper trading`

Do **not** send Kronos' raw forecast directly to a live broker.